# Train on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marcofer14/GraphProject/blob/main/notebooks/colab_train.ipynb)

Runs any training config from `configs/` on a free Colab GPU. Checkpoints and logs go to Google
Drive, so if the session disconnects you re-run all cells and training **resumes** from the last
epoch.

**Before starting:** *Runtime → Change runtime type → T4 GPU*.

At the end, download `<CONFIG>_colab.zip` from `MyDrive/neural-routing/` and unzip it at the
repository root: it contains `checkpoints/` and `runs/` in the layout the benchmarks expect.

In [ ]:
# Which experiment to run: any file name in configs/ (without .yaml)
CONFIG = "tsp20_rollout"
EXTRA_OVERRIDES = ""  # e.g. "train.epochs=50"

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys

import torch

print(sys.version.split()[0], "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable the GPU: Runtime -> Change runtime type -> T4 GPU"

## 2 · Google Drive (persistent checkpoints)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/neural-routing"
!mkdir -p {OUT}/checkpoints {OUT}/runs

## 3 · Install

`pip` (not `uv`) on purpose: `pyproject.toml` maps Linux to the CPU build of PyTorch for CI, and
pip ignores that mapping, keeping Colab's preinstalled CUDA build.

In [ ]:
import os
import sys

if not os.path.exists("/content/GraphProject"):
    !git clone -q https://github.com/Marcofer14/GraphProject.git /content/GraphProject
%cd /content/GraphProject
!git pull -q
flag = "" if sys.version_info[:2] == (3, 12) else "--ignore-requires-python"
!pip install -q -e . {flag}
!python -c "import torch, neural_routing; print('OK, CUDA =', torch.cuda.is_available())"

## 4 · Train (resumes automatically from Drive)

In [ ]:
!python -m neural_routing.ml.train --config-name {CONFIG} \
    train.device=cuda \
    train.checkpoint_dir={OUT}/checkpoints \
    train.run_dir={OUT}/runs/{CONFIG} {EXTRA_OVERRIDES}

## 5 · Package results

In [ ]:
!cd {OUT} && rm -f {CONFIG}_colab.zip && zip -qr {CONFIG}_colab.zip checkpoints/{CONFIG}*.pt checkpoints/*_rollout*.pt runs/{CONFIG} 2>/dev/null; true
!ls -lh {OUT}/{CONFIG}_colab.zip
!tail -n 3 {OUT}/runs/{CONFIG}/train.log